# quantlab on a T4: FP16 vs GPTQ-Int8 vs GPTQ-Int4 vs AWQ for Qwen2.5-3B-Instruct

For each scheme: launch vLLM, read weights and KV cache from the startup log, measure perplexity (200 WikiText-2 paragraphs via prompt logprobs) and GSM8K exact match (100 problems, greedy), then a tokenbench closed-loop sweep at 1, 16 and 64 users. ~10 min install + ~8 min per scheme.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# Install recipe from inference-infra-portfolio/KAGGLE.md
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/quantlab" "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, quantlab.report, servellm.launch, tokenbench
assert torch.cuda.is_available()
import vllm.multimodal.media.audio
print("vllm", vllm.__version__, "torch", torch.__version__)

## Fixed eval sets (same for every scheme)

In [ ]:
# Read the eval sets straight from their parquet files on the Hub (run 1 died on the `datasets`
# short name "wikitext": the Hub client now requires namespace/name ids).
import pandas as pd
HF = "https://huggingface.co/datasets"
wt = pd.read_parquet(f"{HF}/Salesforce/wikitext/resolve/main/wikitext-2-raw-v1/test-00000-of-00001.parquet")
texts = [t.strip() for t in wt["text"] if len(t.split()) >= 60 and not t.strip().startswith("=")][:200]
texts = [" ".join(t.split()[:350]) for t in texts]       # cap length so each fits comfortably in context
gsm = pd.read_parquet(f"{HF}/openai/gsm8k/resolve/main/main/test-00000-of-00001.parquet").head(100)
items = [{"question": q, "answer": a} for q, a in zip(gsm["question"], gsm["answer"])]
print(len(texts), "ppl texts;", len(items), "gsm8k items")

## One function per scheme

In [ ]:
import asyncio, json, subprocess, httpx, time
from servellm.launch import ServerConfig, VllmServer
from quantlab.evals import measure_perplexity, measure_gsm8k

SCHEMES = [
    ("fp16",      "Qwen/Qwen2.5-3B-Instruct",           ()),
    ("gptq-int8", "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int8", ()),
    ("gptq-int4", "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4", ()),
    ("awq",       "Qwen/Qwen2.5-3B-Instruct-AWQ",       ()),
]   # vLLM reads the quantization method from each checkpoint's config; no --quantization flag needed

async def quality(model):
    async with httpx.AsyncClient(base_url="http://127.0.0.1:8000", timeout=None) as c:
        ppl = await measure_perplexity(c, model, texts)
        g = await measure_gsm8k(c, model, items)
    return {**ppl, **g}

def run_sync(coro):
    # Jupyter already runs an event loop, so asyncio.run() raises here (run 2); run the
    # coroutine on a worker thread with its own loop instead.
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, coro).result()

def run_scheme(name, model, extra):
    out = f"results/{name}.json"
    server = VllmServer(ServerConfig(model=model, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9, extra_args=extra),
                        log_path=f"results/startup_{name}.log")
    try:
        info = server.start(ready_timeout_s=1200)
        rec = {"scheme": name, "model": model, "startup": {"model_load_gib": info.model_load_gib, "model_load_seconds": info.model_load_seconds,
               "kv_cache_gib": info.kv_cache_gib, "kv_cache_tokens": info.kv_cache_tokens, "attention_backend": info.attention_backend}}
        t0 = time.time(); rec["quality"] = run_sync(quality(model)); rec["quality"]["seconds"] = time.time() - t0
        print(name, "quality:", {k: v for k, v in rec["quality"].items() if k not in ("hits", "examples")})
        subprocess.run(["tokenbench", "sweep", "--base-url", "http://127.0.0.1:8000", "--model", model, "--out", f"results/sweep_{name}.json",
                        "--levels", "1,16,64", "--num-requests", "128", "--warmup", "4", "--prompt-tokens", "256", "--max-tokens", "128"], check=True)
        sw = json.load(open(f"results/sweep_{name}.json"))
        rec["latency"] = {str(int(l["level"])): {"ttft_p50_ms": l["ttft"]["p50"] * 1000, "itl_p50_ms": l["itl"]["p50"] * 1000,
                          "itl_p95_ms": l["itl"]["p95"] * 1000, "output_tokens_per_s": l["output_tokens_per_s"]} for l in sw["levels"]}
        # which kernel did vLLM pick? grep the startup log for quantization lines
        rec["startup"]["quant_log_lines"] = [l.strip()[:200] for l in open(f"results/startup_{name}.log") if "quant" in l.lower() or "marlin" in l.lower() or "exllama" in l.lower()][:8]
    except Exception as e:
        rec = {"scheme": name, "model": model, "error": f"{type(e).__name__}: {e}"}
        print(name, "FAILED:", rec["error"])
    finally:
        server.stop(); time.sleep(5)
    json.dump(rec, open(out, "w"), indent=1)
    return rec

runs = [run_scheme(*s) for s in SCHEMES]

## Comparison

In [ ]:
from quantlab.report import build_table, render_markdown
from IPython.display import Markdown, display
ok = [r for r in runs if "error" not in r]
rows = build_table(ok)
md = render_markdown(rows)
open("results/comparison.md", "w").write(md); json.dump(rows, open("results/comparison.json", "w"), indent=1)
display(Markdown(md))
for r in runs:
    if "error" in r: print("FAILED:", r["scheme"], r["error"])

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.2))
for r in ok:
    xs = sorted(int(k) for k in r["latency"]); ax.plot(xs, [r["latency"][str(x)]["itl_p50_ms"] for x in xs], marker="o", label=r["scheme"])
ax.set_xscale("log", base=2); ax.set_xticks([1, 16, 64]); ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.set_xlabel("concurrent requests"); ax.set_ylabel("ITL p50 (ms)"); ax.set_title("Decode latency per scheme vs concurrency (Qwen2.5-3B, T4)"); ax.grid(alpha=.3); ax.legend()
fig.tight_layout(); fig.savefig("results/itl_vs_concurrency.png", dpi=130)
from IPython.display import Image; display(Image("results/itl_vs_concurrency.png"))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip